# KRM-Core: GPU'da Kavramsal Parçalı Eğitim

RWKV tabanlı, tüketici donanımında büyük model eğitimi.

**Yapılacaklar:**
1. Repo'yu klonla
2. Bağımlılıkları kur
3. Eğitim verisini oluştur
4. RWKV modelini GPU'da eğit
5. Modeli Drive'a kaydet


In [ ]:
# 1. Repo'yu klonla
import os, sys, json, time, gc
from pathlib import Path

!git clone https://github.com/asosyal04440/KRM-Core.git
%cd KRM-Core
!git pull

In [ ]:
# 2. Bağımlılıkları kur
!pip install torch numpy tqdm

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Eğitim Verisi

Büyük spec'ler (Intel SDM 13MB) repo'da yok. İki seçenek:

**A) Otomatik indir:** Intel SDM resmi PDF'ten text çıkar (internet gerekir)

**B) Drive'dan yükle:** Kendi spec dosyalarını Google Drive'a koy

Aşağıdaki hücre A seçeneğini çalıştırır.

In [ ]:
# 3a. Intel SDM'yi indir ve işle (resmi Intel PDF -> metin)
import urllib.request
import zipfile

# Intel SDM resmi PDF linki (Haziran 2025)
SDM_URL = "https://www.intel.com/content/www/us/en/content-details/325462/"

print("Intel SDM indiriliyor...")
# Not: Intel SDM büyük (~50MB PDF).
# Gerçek indirme için pdfkit/pymupdf gerekir.
# Alternatif: Önceden işlenmiş .md dosyasını Drive'a koy.

print("\nAlternatif: Spec dosyalarını Drive'a yükleyip bağlayın:")
print("from google.colab import drive")
print("drive.mount('/content/drive')")

In [ ]:
# 3b. Repo'daki verilerden corpus oluştur
import sys
sys.path.insert(0, '.')

# Spec->code çiftlerini topla
corpus_parts = []

# Training corpus
corpus_dir = Path('data/training_corpus')
for f in corpus_dir.glob('*.jsonl'):
    if 'intel_sdm' in f.name or 'mega_spec' in f.name:
        continue  # çok büyük, atla
    with open(f, 'r', encoding='utf-8') as fh:
        for line in fh:
            try:
                d = json.loads(line)
                inp = d.get('input', {})
                tgt = d.get('target', {})
                spec = inp.get('spec_context', inp.get('spec', ''))
                req = inp.get('requirement', '')
                code = tgt.get('code', '')
                if isinstance(code, dict):
                    code = '\n'.join(code.values())
                if isinstance(code, list):
                    code = '\n'.join(code)
                corpus_parts.append(f"# SPEC: {spec}\n{req}\n```rust\n{code[:2000]}\n```")
            except: pass

# Concepts
concepts_path = Path('data/echos_concepts/all_concepts.json')
if concepts_path.exists():
    concepts = json.loads(concepts_path.read_text('utf-8'))
    for c in concepts[:200]:
        name = c.get('concept', c.get('name', ''))
        desc = c.get('description', c.get('definition', ''))
        if name:
            corpus_parts.append(f"# CONCEPT: {name}\n{desc}")

corpus_text = '\n\n---\n\n'.join(corpus_parts)
print(f"Corpus: {len(corpus_text):,} chars (~{len(corpus_text)//4:,} tokens)")

output_path = Path('data/rwkv_training')
output_path.mkdir(parents=True, exist_ok=True)
output_path.joinpath('echos_colab_corpus.txt').write_text(corpus_text, 'utf-8')

In [ ]:
# 3c. Google Drive'a bağlan (büyük spec'ler için)
from google.colab import drive
drive.mount('/content/drive')

# Drive'da spec varsa kullan
drive_specs = Path('/content/drive/MyDrive/echos_specs')
if drive_specs.exists():
    print("Drive'da spec bulundu!")
    for f in drive_specs.glob('*.md'):
        print(f"  {f.name} ({f.stat().st_size/1e6:.1f}MB)")
        text = f.read_text('utf-8', errors='replace')
        # Ana corpus'a ekle
        existing = output_path.joinpath('echos_colab_corpus.txt').read_text('utf-8')
        output_path.joinpath('echos_colab_corpus.txt').write_text(
            existing + f"\n\n# === {f.name} ===\n{text}", 'utf-8')
    print("Corpus güncellendi!")

## 4. RWKV Model Eğitimi (GPU)

Kavramsal Parçalı Eğitim ile RWKV modelini eğit.
Blok blok eğit - her blok RAM'e sığar.

In [ ]:
# 4. Modeli eğit

class CharTokenizer:
    def __init__(self):
        self.vocab_size = 256
        self.pad_token = 0
        self.bos_token = 1
        self.eos_token = 2
        self.unk_token = 3

    def encode(self, text):
        tokens = [self.bos_token]
        for ch in text:
            b = ord(ch)
            tokens.append(b if b < 256 else self.unk_token)
        tokens.append(self.eos_token)
        return tokens

    def decode(self, tokens):
        chars = []
        for t in tokens:
            if t >= 4 and t < 256:
                chars.append(chr(t))
            elif t == self.eos_token:
                break
        return "".join(chars)

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

# RWKV model (ConceptBlock)
sys.path.insert(0, 'scripts')
from train_conceptual_sharded import ConceptBlock

# Veri
corpus = output_path.joinpath('echos_colab_corpus.txt').read_text('utf-8', errors='replace')
tokenizer = CharTokenizer()
tokens = tokenizer.encode(corpus)
print(f"Tokens: {len(tokens):,}")

# Dataset
class SimpleDataset(Dataset):
    def __init__(self, tokens, seq_len=256, num_samples=5000):
        self.seq_len = seq_len
        self.tokens = tokens
        stride = max(1, (len(tokens) - seq_len) // num_samples)
        self.num_samples = min(num_samples, max(1, (len(tokens) - seq_len) // stride))
        self.stride = stride
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        start = idx * self.stride
        chunk = self.tokens[start:start + self.seq_len]
        if len(chunk) < self.seq_len:
            chunk = chunk + [0] * (self.seq_len - len(chunk))
        return torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)

# Hyperparametreler
D_MODEL = 256       # Artırılabilir (512, 768...)
BATCH_SIZE = 32     # GPU'da 32+ rahat
SEQ_LEN = 256
EPOCHS = 20
LR = 3e-4
NUM_BLOCKS = 1      # Her spec için 1 blok

dataset = SimpleDataset(tokens, SEQ_LEN, num_samples=5000)
val_size = int(len(dataset) * 0.05)
train_ds, val_ds = torch.utils.data.random_split(dataset, [len(dataset)-val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

block = ConceptBlock('echos_block', vocab_size=256, d_model=D_MODEL, n_layers=2, d_ff=D_MODEL*4)
block.to(device)
params = sum(p.numel() for p in block.parameters())
print(f"Params: {params:,}")
print(f"GPU Mem: {params*4/1024/1024:.1f} MB")

optim = torch.optim.AdamW(block.parameters(), lr=LR, weight_decay=0.01)
history = {"train_loss": [], "val_loss": []}
best_loss = float('inf')

print("\nTraining...")
for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    block.train()
    total_loss = 0
    total_tokens = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits, loss = block(x, y)
        optim.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(block.parameters(), 1.0)
        optim.step()
        total_loss += loss.item()
        total_tokens += y.numel()
    
    # Val
    block.eval()
    val_loss = 0
    val_tokens = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = block(x, y)
            val_loss += loss.item()
            val_tokens += y.numel()
    
    avg_train = total_loss / total_tokens
    avg_val = val_loss / val_tokens
    ppl = torch.exp(torch.tensor(avg_val)).item()
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    
    print(f"Ep {epoch+1:2d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | PPL: {ppl:.2f} | {time.time()-t0:.1f}s")
    
    if avg_val < best_loss:
        best_loss = avg_val
        torch.save({'state': block.state_dict(), 'config': {'d_model': D_MODEL}},
                   'best_model.pt')

print(f"\nBest val loss: {best_loss:.4f}")
print(f"Best PPL: {torch.exp(torch.tensor(best_loss)).item():.2f}")

In [ ]:
# 5. Örnek üret
block.eval()
seed = torch.randint(4, 256, (1, 1), device=device)
gen = seed.tolist()[0]

with torch.no_grad():
    for _ in range(500):
        logits, _ = block(seed, None)
        probs = F.softmax(logits[0, -1] / 0.8, dim=0)
        t = torch.multinomial(probs, 1).item()
        gen.append(t)
        seed = torch.cat([seed, torch.tensor([[t]], device=device)], dim=1)[:, -SEQ_LEN:]

print(tokenizer.decode(gen))

In [ ]:
# 6. Modeli Google Drive'a kaydet
save_path = Path('/content/drive/MyDrive/KRM-Core')
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state': block.state_dict(),
    'history': history,
    'config': {'d_model': D_MODEL, 'epochs': EPOCHS, 'params': params},
    'best_val_loss': best_loss
}, save_path / 'echos_model.pt')

# Ayrıca best model
import shutil
if Path('best_model.pt').exists():
    shutil.copy('best_model.pt', save_path / 'best_model.pt')

print(f"Model kaydedildi: {save_path}/")
print(f"Dosyalar: {[f.name for f in save_path.glob('*')]}")

## Sonraki Adımlar

1. **Çoklu blok eğitimi**: Her spec concept'i için ayrı blok
2. **BPE Tokenizer**: Karakter yerine gerçek tokenizer
3. **400B ölçeklendirme**: 449 blok × 890M parametre
4. **Rezonans Motoru**: Eğitilmiş blokları KRM-Core ile birleştir

Daha fazla veri için `scripts/build_raw_corpus.py` çalıştırılabilir.